# From Voice to Vision — 5. Enhanced Training Pipeline

Following the diagnosis of the first optimisation study, the pipeline is strengthened along four
axes before the search is repeated:

* the validation set is enlarged from two to **four speakers**, reducing the noise in the fitness;
* the fitness becomes the **mean of the three best validation epochs** rather than a single peak;
* **SpecAugment** masks random bands of time and frequency during training, discouraging the network
  from over-relying on narrow spectro-temporal cues;
* the spectrogram is stacked with its **first and second temporal derivatives**, giving the model
  explicit access to prosodic dynamics, and a learning-rate schedule stabilises training.

In [ ]:
# Clone the project repository and install the audio dependencies
REPO_URL = "https://github.com/Nadaa3672/from-voice-to-vision.git"
import os
repo = REPO_URL.rstrip("/").split("/")[-1].replace(".git", "")
if not os.path.exists(repo):
    !git clone $REPO_URL
%cd $repo
!git pull -q
!pip install -q librosa soundfile noisereduce tqdm

In [ ]:
from src import config, data_loader, features
import numpy as np
data_loader.download_ravdess()
df = data_loader.build_index()
data = features.build_dataset(df, denoise=False, cache=True)
data["split"] = df["split"].to_numpy()
splits = features.split_arrays(data)
print("Split sizes:", {s: len(splits[s]["y"]) for s in ["train", "val", "test"]})

## Effect of the enhanced pipeline alone

Before re-running the search, the enhanced pipeline is evaluated with the default
hyper-parameters, to separate the contribution of the pipeline from that of the optimisation.

In [ ]:
from src.models import cnn
print("Device:", cnn.get_device())

hp_default = {"lr": 1e-3, "dropout": 0.3, "weight_decay": 1e-4,
              "batch_size": 32, "width": 32}
check = cnn.train_cnn(splits, hp_default, epochs=45, patience=12,
                      deltas=True, augment=True, verbose=True)
print(f"\nEnhanced pipeline, default hyper-parameters → test = {check['test_acc']:.3f}")

## Re-running the search on the enhanced pipeline

In [ ]:
QUICK = False
if QUICK:
    N_AGENTS, N_ITER, EPOCHS_SEARCH = 5, 3, 8
else:
    N_AGENTS, N_ITER, EPOCHS_SEARCH = 6, 6, 14

from src.optimization import pso, fso, ga
from src.optimization.search_space import make_objective, decode, DIM
import time

def run(optimizer, **kw):
    obj = make_objective(splits, epochs=EPOCHS_SEARCH, patience=5)
    t0 = time.time()
    res = optimizer.optimize(obj, DIM, seed=config.SEED, **kw)
    res["time_s"] = time.time() - t0
    res["n_evals"] = len(obj.history_evals)
    res["best_hp"] = decode(res["best_u"])
    print(f"{res['name']}: robust validation = {res['best_fit']:.3f} | "
          f"evaluations = {res['n_evals']} | time = {res['time_s']:.0f} s")
    return res

In [ ]:
res_pso = run(pso, n_particles=N_AGENTS, n_iter=N_ITER); print(res_pso["best_hp"])

In [ ]:
res_fso = run(fso, n_particles=N_AGENTS, n_iter=N_ITER); print(res_fso["best_hp"])

In [ ]:
res_ga = run(ga, pop_size=N_AGENTS, n_iter=N_ITER); print(res_ga["best_hp"])

In [ ]:
import matplotlib.pyplot as plt, pandas as pd
allres = [res_pso, res_fso, res_ga]
plt.figure(figsize=(9, 5))
for r in allres:
    plt.plot(range(len(r["history"])), r["history"], marker="o", label=r["name"])
plt.xlabel("iteration"); plt.ylabel("robust validation accuracy")
plt.title("Convergence on the enhanced pipeline"); plt.legend(); plt.grid(alpha=.3)
plt.tight_layout()
config.FIGURES_DIR.mkdir(parents=True, exist_ok=True)
plt.savefig(config.FIGURES_DIR / "04_convergence.png", dpi=150)
plt.show()

display(pd.DataFrame([{"algorithm": r["name"], "robust_val": round(r["best_fit"], 3),
                       "evaluations": r["n_evals"], "time_s": round(r["time_s"]),
                       **r["best_hp"]} for r in allres]))

The enhanced pipeline improves generalisation appreciably. The search itself, however, still
selects its winner on a validation signal that correlates imperfectly with test performance, which
motivates the seeded protocol and the ensemble developed in the next notebook.